#Installation of datasets

In [ ]:
!pip install datasets --quiet

#Dataset Summary
BIGPATENT, consisting of 1.3 million records of U.S. patent documents along with human written abstractive summaries. Each US patent application is filed under a Cooperative Patent Classification (CPC) code. There are nine such classification categories:

a: Human Necessities
b: Performing Operations; Transporting
c: Chemistry; Metallurgy
d: Textiles; Paper
e: Fixed Constructions
f: Mechanical Engineering; Lightning; Heating; Weapons; Blasting
g: Physics
h: Electricity
y: General tagging of new or cross-sectional technology
currently

In [ ]:
from datasets import load_dataset
ds = load_dataset("big_patent","y",download_mode="force_redownload",trust_remote_code=True)


In [15]:
ds

DatasetDict({
    train: Dataset({
        features: ['description', 'abstract'],
        num_rows: 124397
    })
    validation: Dataset({
        features: ['description', 'abstract'],
        num_rows: 6911
    })
    test: Dataset({
        features: ['description', 'abstract'],
        num_rows: 6911
    })
})

In [16]:
# Export the splits to CSV files
ds['train'].to_csv('train_data.csv', index=False)
ds['validation'].to_csv('validation_data.csv', index=False)
ds['test'].to_csv('test_data.csv', index=False)

Creating CSV from Arrow format:   0%|          | 0/125 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

204379055

In [17]:
!pip install  -U langchain-community sentence-transformers --quiet

In [18]:
# === 1. Import Necessary Libraries ===
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import random

In [19]:
 #=== 2. Create Document Objects from the Dataset ===
# For demonstration, we'll use the "train" split.
# Each document is a combination of the "description" and "abstract" fields.
documents = []
for i, record in enumerate(ds["train"]):
    # Combine the fields; you can adjust the format as needed.
    content = f"Description: {record['description']}\nAbstract: {record['abstract']}"
    #print(content)
    metadata = {"doc_id": i}  # Storing a doc_id for reference (you can add more metadata if needed)
    documents.append(Document(page_content=content, metadata=metadata))

print(f"Created {len(documents)} Document objects from the dataset.")

Created 124397 Document objects from the dataset.


In [13]:
# === 3. Create Document Embeddings and Build a FAISS Vectorstore ===
# Here we use the "all-mpnet-base-v2" model for embedding generation.
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# Build the FAISS vector store using the generated embeddings.
vectorstore = FAISS.from_documents(documents, embeddings)
print(f"Indexed {len(documents)} documents in the FAISS vectorstore.")

<ipython-input-13-411c16c767e3>:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

In [ ]:

# === 4. Set Up the Generation Module ===
# Choose a model for generation. Here, we use "microsoft/phi-4".
# (You can substitute with "meta-llama/Meta-Llama-3.1-8B-Instruct" if preferred.)
model_name = "microsoft/phi-4"

# Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Create a text-generation pipeline.
# If you have a GPU available, you can set device=0; otherwise, remove the device parameter.
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [ ]:
 #=== 5. Define the Retrieval-Augmented Generation (RAG) Function ===
def answer_query(query: str, vectorstore, generator, top_k: int = 5, max_length: int = 512):
    """
    Given a user query, retrieve relevant patent documents from the vectorstore
    and generate an answer using the generation model.

    Parameters:
        query (str): The user-provided query.
        vectorstore: The FAISS vectorstore built from the patent documents.
        generator: The text-generation pipeline.
        top_k (int): Number of top documents to retrieve.
        max_length (int): Maximum token length for generated answer.

    Returns:
        answer (str): The generated answer.
        retrieved_docs (list): List of retrieved Document objects.
    """
    # --- Retrieval Phase ---
    retrieved_docs = vectorstore.similarity_search(query, k=top_k)

    # Combine the content of the retrieved documents.
    # Here we include the doc_id for reference. You can adjust as needed.
    context = "\n\n".join([
        f"Doc ID: {doc.metadata.get('doc_id', 'N/A')}\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    # --- Generation Phase ---
    # Construct the prompt including the context and the user query.
    prompt = (
        "You are an Intellectual Property Assistant that helps analyze patent documents. "
        "Based on the following patent excerpts, answer the query below. Be sure to reference relevant details from the patents.\n\n"
        f"Patent Excerpts:\n{context}\n\n"
        f"Query: {query}\n"
        "Answer:"
    )

    # Generate the answer using the text-generation pipeline.
    output = generator(prompt, max_length=max_length, do_sample=False)
    answer = output[0]['generated_text']

    return answer, retrieved_docs

In [ ]:

# === 6. Test the RAG Pipeline with a Sample Query ===
sample_query = "Find patents related to innovative solar panel technologies that discuss photovoltaic materials."
answer, docs = answer_query(sample_query, vectorstore, generator)

print("\n=== Generated Answer ===")
print(answer)

print("\n=== Retrieved Document IDs ===")
for doc in docs:
    print(doc.metadata.get("doc_id"))

In [20]:
!pip install langchain faiss-cpu transformers sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 60.4 MB/s eta 0:00:00


In [21]:
!pip install  -U langchain-community --quiet
#https://huggingface.co/sentence-transformers/all-mpnet-base-v2 --embedding model

In [14]:
#corrected
import os
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Function to load patent documents from a directory.
def load_patent_documents(directory_path: str):
    documents = []
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path) and filename.endswith(".txt"):
            with open(file_path, "r", encoding="utf-8") as file:
                content = file.read().strip()  # Remove extra whitespace
                if content:  # Only add non-empty documents
                    documents.append(Document(page_content=content, metadata={"source": filename}))
    return documents

# Load documents from the directory.
patent_docs = load_patent_documents("/content")

# Check if any documents were loaded before proceeding.
if not patent_docs:
    raise ValueError("No patent documents found. Check the directory path and ensure .txt files have content.")

# Create embeddings for the documents using a transformer model.
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# Build a FAISS vector store from the documents.
vectorstore = FAISS.from_documents(patent_docs, embeddings)

print(f"Indexed {len(patent_docs)} patent documents.")


ValueError: No patent documents found. Check the directory path and ensure .txt files have content.

In [22]:
"""import os
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Function to load patent documents from a directory.
def load_patent_documents(directory_path: str):
    documents = []
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path) and filename.endswith(".txt"):
            with open(file_path, "r", encoding="utf-8") as file:
                content = file.read()
                # Optionally, you could parse and label different sections here.
                documents.append(Document(page_content=content, metadata={"source": filename}))
    return documents

# Load documents from the BigPatent directory.
patent_docs = load_patent_documents("/content")

# Create embeddings for the documents using a transformer model.
# You can choose any suitable embedding model (here we use "all-mpnet-base-v2" as an example).
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# Build a FAISS vector store from the documents.
vectorstore = FAISS.from_documents(patent_docs, embeddings)

print(f"Indexed {len(patent_docs)} patent documents.")
"""

'import os\nfrom langchain.docstore.document import Document\nfrom langchain.embeddings import HuggingFaceEmbeddings\nfrom langchain.vectorstores import FAISS\n\n# Function to load patent documents from a directory.\ndef load_patent_documents(directory_path: str):\n    documents = []\n    for filename in os.listdir(directory_path):\n        file_path = os.path.join(directory_path, filename)\n        if os.path.isfile(file_path) and filename.endswith(".txt"):\n            with open(file_path, "r", encoding="utf-8") as file:\n                content = file.read()\n                # Optionally, you could parse and label different sections here.\n                documents.append(Document(page_content=content, metadata={"source": filename}))\n    return documents\n\n# Load documents from the BigPatent directory.\npatent_docs = load_patent_documents("/content")\n\n# Create embeddings for the documents using a transformer model.\n# You can choose any suitable embedding model (here we use "all

In [ ]:
# === 1. Import Necessary Libraries ===
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import random

# Assuming your Hugging Face dataset 'ds' is already loaded and has the following structure:
# ds = DatasetDict({
#     "train": Dataset(... features: ['description', 'abstract'], num_rows: 124397),
#     "validation": Dataset(...),
#     "test": Dataset(...),
# })

# === 2. Create Document Objects from the Dataset ===
# For demonstration, we'll use the "train" split.
# Each document is a combination of the "description" and "abstract" fields.
documents = []
for i, record in enumerate(ds["train"]):
    # Combine the fields; you can adjust the format as needed.
    content = f"Description: {record['description']}\nAbstract: {record['abstract']}"
    metadata = {"doc_id": i}  # Storing a doc_id for reference (you can add more metadata if needed)
    documents.append(Document(page_content=content, metadata=metadata))

print(f"Created {len(documents)} Document objects from the dataset.")

# === 3. Create Document Embeddings and Build a FAISS Vectorstore ===
# Here we use the "all-mpnet-base-v2" model for embedding generation.
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# Build the FAISS vector store using the generated embeddings.
vectorstore = FAISS.from_documents(documents, embeddings)
print(f"Indexed {len(documents)} documents in the FAISS vectorstore.")

# === 4. Set Up the Generation Module ===
# Choose a model for generation. Here, we use "microsoft/phi-4".
# (You can substitute with "meta-llama/Meta-Llama-3.1-8B-Instruct" if preferred.)
model_name = "microsoft/phi-4"

# Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Create a text-generation pipeline.
# If you have a GPU available, you can set device=0; otherwise, remove the device parameter.
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

# === 5. Define the Retrieval-Augmented Generation (RAG) Function ===
def answer_query(query: str, vectorstore, generator, top_k: int = 5, max_length: int = 512):
    """
    Given a user query, retrieve relevant patent documents from the vectorstore
    and generate an answer using the generation model.

    Parameters:
        query (str): The user-provided query.
        vectorstore: The FAISS vectorstore built from the patent documents.
        generator: The text-generation pipeline.
        top_k (int): Number of top documents to retrieve.
        max_length (int): Maximum token length for generated answer.

    Returns:
        answer (str): The generated answer.
        retrieved_docs (list): List of retrieved Document objects.
    """
    # --- Retrieval Phase ---
    retrieved_docs = vectorstore.similarity_search(query, k=top_k)

    # Combine the content of the retrieved documents.
    # Here we include the doc_id for reference. You can adjust as needed.
    context = "\n\n".join([
        f"Doc ID: {doc.metadata.get('doc_id', 'N/A')}\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    # --- Generation Phase ---
    # Construct the prompt including the context and the user query.
    prompt = (
        "You are an Intellectual Property Assistant that helps analyze patent documents. "
        "Based on the following patent excerpts, answer the query below. Be sure to reference relevant details from the patents.\n\n"
        f"Patent Excerpts:\n{context}\n\n"
        f"Query: {query}\n"
        "Answer:"
    )

    # Generate the answer using the text-generation pipeline.
    output = generator(prompt, max_length=max_length, do_sample=False)
    answer = output[0]['generated_text']

    return answer, retrieved_docs

# === 6. Test the RAG Pipeline with a Sample Query ===
sample_query = "Find patents related to innovative solar panel technologies that discuss photovoltaic materials."
answer, docs = answer_query(sample_query, vectorstore, generator)

print("\n=== Generated Answer ===")
print(answer)

print("\n=== Retrieved Document IDs ===")
for doc in docs:
    print(doc.metadata.get("doc_id"))


Created 124397 Document objects from the dataset.


In [ ]:
# %% [markdown]
# # IPAdvisor: An Intellectual Property AI Assistant
#
# This notebook demonstrates an end-to-end Retrieval-Augmented Generation (RAG) system for patent management using the BigPatent dataset.
# It performs:
# - Document indexing and semantic retrieval (using LlamaIndex)
# - Answer generation using a state-of-the-art LLM (Meta-Llama-3.1-8B-Instruct)
# - Patent-specific tasks: key section extraction, similarity analysis, and evaluation.
#
# Note: Adjust the model identifiers and dependencies as needed.

# %% [code]
# Install dependencies. (Uncomment and run if not already installed)
!pip install llama-index langchain faiss-cpu transformers sentence-transformers streamlit

# %% [markdown]
# ## 1. Data Indexing & Retrieval
#
# We use the BigPatent dataset from Hugging Face. For simplicity, this demo assumes a local folder `./big_patent_data` containing patent documents.
#
# In practice, you may need to download and preprocess the dataset. Here, we assume documents are stored as text files.

# %% [code]
import os
from llama_index import SimpleDirectoryReader, VectorStoreIndex
from sentence_transformers import SentenceTransformer

# Folder containing patent text files (each file is one patent document)
DATA_PATH = './big_patent_data'

# Load documents using LlamaIndex's reader
documents = SimpleDirectoryReader(DATA_PATH).load_data()
print(f"Loaded {len(documents)} patent documents.")

# %% [markdown]
# ### Build the Vector Index
#
# We use LlamaIndex to compute embeddings and build a vector store. Adjust parameters as needed.

# %% [code]
# Build a vector index (this uses an underlying FAISS index by default)
index = VectorStoreIndex.from_documents(documents)
print("Index built.")

# %% [markdown]
# ## 2. Natural Language Generation Setup
#
# We load a state-of-the-art open-source model using Hugging Face Transformers. This example uses Meta-Llama-3.1-8B-Instruct.
# Ensure you have access to the model or adjust with an alternative (e.g., microsoft/phi-4).

# %% [code]
import torch
import transformers

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # Change if necessary
generator = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)
print("LLM loaded.")

# %% [markdown]
# ## 3. Define the RAG Workflow
#
# The workflow includes:
# 1. Accepting a user query.
# 2. Retrieving the top-5 relevant patent documents.
# 3. Constructing a prompt that includes context from these documents.
# 4. Generating an answer from the LLM.

# %% [code]
def retrieve_patent_context(query, top_k=5):
    """Retrieve top-k patent documents for a given query."""
    retrieved_docs = index.query(query, similarity_top_k=top_k)
    context = "\n\n".join([doc.text for doc in retrieved_docs.response.docs])
    return context, retrieved_docs.response.docs

def generate_answer(query, context):
    """Generate an answer using the LLM given the query and context."""
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    output = generator(prompt, max_new_tokens=256, do_sample=False)
    return output[0]['generated_text']

# %% [markdown]
# ## 4. Patent Analysis Functions
#
# We include functions to extract key sections and analyze similarity. In a production system, you might need to use
# regex or more advanced parsing to extract claims, abstract, etc.

# %% [code]
def extract_key_sections(document_text):
    """
    Dummy extraction function.
    In practice, implement rules or an LLM prompt to extract:
      - Claims
      - Abstract
      - Drawings (if available)
    Here, we assume the first 200 words are the abstract and the next 300 as claims.
    """
    words = document_text.split()
    abstract = " ".join(words[:200])
    claims = " ".join(words[200:500])
    return abstract, claims

def compute_similarity(new_patent_text, retrieved_texts):
    """
    Compute a simple similarity metric between a new patent text and a list of retrieved texts.
    This demo uses SentenceTransformer to embed texts and compute cosine similarity.
    """
    model = SentenceTransformer('all-MiniLM-L6-v2')
    new_embed = model.encode(new_patent_text)
    sims = []
    for text in retrieved_texts:
        embed = model.encode(text)
        cos_sim = (new_embed @ embed) / (np.linalg.norm(new_embed) * np.linalg.norm(embed))
        sims.append(cos_sim)
    return sims

# %% [code]
import numpy as np
# %% [markdown]
# ## 5. Evaluation Framework (Simplified Demo)
#
# Here we define a simple evaluation routine that:
# - Accepts a set of queries (for demonstration, a list of sample queries)
# - For each query, retrieves context, generates an answer, and prints the output.
#
# In a full study, results would be exported for human evaluation (e.g. using Appen).

# %% [code]
sample_queries = [
    "Find prior art related to wireless charging for electric vehicles.",
    "Summarize the key claims in this patent on drone delivery systems.",
    "Identify similarities between patent US1234567 and new application on AR headsets."
]

for query in sample_queries:
    print(f"\n=== Query: {query} ===")
    context, docs = retrieve_patent_context(query)
    answer = generate_answer(query, context)
    print("\nGenerated Answer:\n", answer)
    print("\n--- Retrieved Patent Snippets ---")
    for i, doc in enumerate(docs, 1):
        print(f"Snippet {i}: {doc.text[:300]}...")  # Print first 300 chars for brevity

# %% [markdown]
# ## 6. Interactive Interface (Optional)
#
# You can use Streamlit or Gradio to create an interactive interface. Below is a simple Streamlit example.
#
# To run the Streamlit app, save the below code to a file (e.g., `app.py`), and run `!streamlit run app.py` in a Colab cell.

# %% [code]
%%bash
cat << 'EOF' > app.py
import streamlit as st
from llama_index import SimpleDirectoryReader, VectorStoreIndex
import transformers, torch

# Load index (assumes same DATA_PATH and index built as above)
DATA_PATH = './big_patent_data'
documents = SimpleDirectoryReader(DATA_PATH).load_data()
index = VectorStoreIndex.from_documents(documents)

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
generator = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

def retrieve_patent_context(query, top_k=5):
    retrieved_docs = index.query(query, similarity_top_k=top_k)
    context = "\n\n".join([doc.text for doc in retrieved_docs.response.docs])
    return context

def generate_answer(query, context):
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    output = generator(prompt, max_new_tokens=256, do_sample=False)
    return output[0]['generated_text']

st.title("IPAdvisor: Patent Assistant")
user_query = st.text_input("Enter your patent query:")
if user_query:
    context = retrieve_patent_context(user_query)
    answer = generate_answer(user_query, context)
    st.write("### Generated Answer")
    st.write(answer)
    st.write("### Context Retrieved")
    st.write(context)
EOF

#echo "To run the Streamlit app, execute: streamlit run app.py"
# %% [markdown]
# ## 7. Conclusion
#
# This Colab notebook provides a full implementation of the IPAdvisor system in a RAG framework. It covers data indexing, retrieval,
# natural language generation for answering queries, and additional functions for patent analysis and evaluation. Adjust paths, models,
# and extraction functions as needed for your specific use case.
#
# You can now run this notebook to test and refine the system.

